# 仮説
Spatial LoRA が LIBERO-Spatial（および Advanced サブセット）で Base より成功率を上げる。

# 変更点（ファイル/パラメータ）
- プロファイル: `spatial` / `advanced` / `all`（`notebooks/configs/profiles/`）
- コード: `notebooks/lib/libero_eval_compare/`
- モデルパスは `configs/profiles/default.yaml` の `models`

# 成功条件
比較 CSV が出力され、suite ごとに Base vs LoRA の成功率が出る。

# 結果（後で追記）
pass / fail / 数値 — 実行後に記入。

# 次
うまくいった評価手順だけを `submit/advanced_colab.ipynb` の該当セクションへ移植する。

---

事前学習済み SmolVLA（LoRA 追加学習なし）と Spatial LoRA ファインチューニング後モデルの成功率を比較します。

- **Baseline**: `workdir/smolvla_libero_plus_baseline`（Advanced Section 8.4）
- **Fine-tuned**: `workdir/smolvla_libero_plus_spatial_lora_merged`（Advanced Section 8.3）

プロファイル:
- `spatial` — LIBERO-Spatial 全 10 タスク × 1 ep（Section 8.6 相当）
- `advanced` — 4 suite × 3 タスク × 1 ep（Section 9 相当）
- `all` — 上記両方

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import torch
from IPython.display import display

# notebooks/experiments -> notebooks/lib
NOTEBOOKS_DIR = Path.cwd().resolve()
if NOTEBOOKS_DIR.name == "experiments":
    NOTEBOOKS_DIR = NOTEBOOKS_DIR.parent
elif NOTEBOOKS_DIR.name != "notebooks":
    candidate = NOTEBOOKS_DIR / "notebooks"
    NOTEBOOKS_DIR = candidate if candidate.is_dir() else NOTEBOOKS_DIR

LIB_DIR = NOTEBOOKS_DIR / "lib"
if str(LIB_DIR) not in sys.path:
    sys.path.insert(0, str(LIB_DIR))

from libero_eval_compare.config import load_profile, list_profiles
from libero_eval_compare.runner import preflight_models

PROFILE = "spatial"  # spatial | advanced | all
SKIP_RUN = True      # True: 既存 eval_info.json を再集計 / False: lerobot-eval を再実行

print("GPU available:", torch.cuda.is_available())
print("Available profiles:", list_profiles())

config = load_profile(PROFILE)
print("Profile:", PROFILE)
print("Baseline:", config.baseline.path)
print("Finetuned:", config.finetuned.path)

preflight_models(config)
print("Preflight OK")

In [ ]:
from libero_eval_compare.compare import build_group_comparison, save_comparison_results
from libero_eval_compare.runner import load_existing_eval_results, run_profile_eval

# Timestamped run dir (do not reuse a fixed notebook_<profile> name)
RUN_DIR = None if SKIP_RUN else config.make_run_dir()

if SKIP_RUN:
    group_results = {}
    for group_name in config.comparison_groups:
        group_config = config.group_config(group_name)
        if PROFILE == "spatial":
            baseline_dir = config.workdir / "eval" / "base"
            finetuned_dir = config.workdir / "eval" / "spatial_lora"
        else:
            baseline_dir = config.workdir / "eval" / "advanced" / "baseline" / group_name
            finetuned_dir = config.workdir / "eval" / "advanced" / "finetuned" / group_name
            if not baseline_dir.is_dir():
                baseline_dir = config.workdir / "eval" / "base"
                finetuned_dir = config.workdir / "eval" / "spatial_lora"

        baseline_results, finetuned_results = load_existing_eval_results(
            baseline_dir,
            finetuned_dir,
            group_config.suites.keys(),
        )
        group_results[group_name] = {
            "baseline": baseline_results,
            "finetuned": finetuned_results,
        }
    source = "notebook_compare_only"
    RUN_DIR = config.make_run_dir()
else:
    eval_output = run_profile_eval(config, run_dir=RUN_DIR, show_progress=True)
    RUN_DIR = eval_output["run_dir"]
    group_results = eval_output["groups"]
    source = "notebook_run"

output_paths = save_comparison_results(
    config,
    run_dir=RUN_DIR,
    group_results=group_results,
    source=source,
)

for group_name, eval_results in group_results.items():
    group_config = config.group_config(group_name)
    comparison_df = build_group_comparison(
        group_config,
        eval_results["baseline"],
        eval_results["finetuned"],
    )
    print(f"\n=== {group_name} ===")
    display(comparison_df.round(1))

print("\nSaved:")
for name, path in output_paths.items():
    print(f"  {name}: {path}")

## CLI から実行する場合

パッケージは `notebooks/lib` にある。リポジトリルートから:

```bash
cd /home/kevin/Matsuo/robot/LIBERO-plus
export PYTHONPATH=notebooks/lib

# 既存結果を再集計（最短）
uv run -m libero_eval_compare compare --profile spatial \
  --baseline-eval workdir/eval/base \
  --finetuned-eval workdir/eval/spatial_lora

# 評価を再実行して比較
uv run -m libero_eval_compare run --profile all
```